# Vehículos involucrados en accidentes de tránsito con víctimas (RUNT, MinTransporte)

Registro de los vehículos involucrados en siniestros de tránsito **con heridos o con muertos**, publicado por el RUNT en cumplimiento de la Ley 2251 de 2022. Trae el tipo, la marca, el modelo y la edad del vehículo, el mes y el lugar del accidente, y su gravedad. Es una buena base para **clasificación con clases desbalanceadas**.

|  |  |
|---|---|
| **Entidad** | Ministerio de Transporte (Registro Único Nacional de Tránsito, RUNT) |
| **Conjunto** | Vehículos involucrados en un accidente de tránsito Ley 2251-2022 · [`6jmc-vaxk`](https://www.datos.gov.co/d/6jmc-vaxk) |
| **Una fila es** | un vehículo involucrado en un accidente con heridos o con muertos |
| **Cobertura** | todo el país, por municipio del accidente |
| **Periodo** | enero de 2022 a diciembre de 2025 (fecha a nivel de mes) |
| **Tamaño total** | 450.335 vehículos (consultado el 31/10/2026); 5,3 % en accidentes con muertos |
| **Actualización** | anual; última carga el 04/09/2026 |
| **Requisitos** | ninguno (token de datos.gov.co opcional) |
| **Tiempo de descarga** | un departamento: segundos · todo el país: 2 a 3 minutos |

**Cómo usar este cuaderno**

1. Lea la ficha y la tabla de variables para saber qué va a descargar.
2. Ajuste la celda **«Parámetros que usted puede ajustar»** (sección 2). Es la única celda que necesita cambiar.
3. Ejecute todo (*Entorno de ejecución → Ejecutar todas* en Colab, o *Run All* en Jupyter). La base queda en la carpeta `datos/`.

Las salidas guardadas en este cuaderno salen de una **muestra pequeña** (los valores por defecto de los parámetros), para que usted vea cómo es la base antes de descargarla completa.

## Qué trae la base



| Columna | Qué es | Tipo |
|---|---|---|
| `tipo_vehiculo` | MOTOCICLETA, AUTOMOVIL, CAMIONETA, BUS, CAMION… | categórica |
| `marca_vehiculo` | Marca | categórica (muchas categorías) |
| `modelo_vehiculo` | Año del modelo | número |
| `edad_vehiculo` | Años desde la matrícula | número |
| `fecha`, `anio`, `mes` | Mes del accidente (viene como `MM/AAAA`) | fecha, número |
| `gravedad_accidente` | CON HERIDOS o CON MUERTOS | categórica |
| `con_muertos` | 1 si el accidente tuvo muertos (calculada) | binaria |
| `departamento_accidente`, `municipio_accidente` | Lugar del accidente (en mayúsculas y sin tildes) | texto |
| `autoridad_de_transito` | Organismo que registró el accidente | categórica |

## Ideas para un modelo de machine learning

| Tarea | Variable objetivo | Predictoras posibles |
|---|---|---|
| Clasificación | `con_muertos` | tipo, marca, edad del vehículo, mes, departamento o municipio |
| Regresión de conteos | vehículos involucrados por municipio y mes | mes, municipio, composición del parque automotor |
| Agrupamiento | municipios según el tipo de vehículos que se accidentan | proporciones por `tipo_vehiculo` |

**Cuidados antes de modelar**

- **Desbalance:** solo 1 de cada 19 vehículos está en un accidente con muertos. La exactitud engaña; use sensibilidad, F1, AUC y `class_weight`.
- La base **no incluye** accidentes solo con daños: el modelo responde «dado que hubo víctimas, ¿hubo muertos?».
- `edad_vehiculo` y `modelo_vehiculo` dicen casi lo mismo: use una de las dos.
- `marca_vehiculo` y `municipio_accidente` tienen cientos de categorías: agrupe las poco frecuentes antes de codificar.

## 1. Instalación

En Colab casi todo viene instalado; esta celda agrega lo que falte.

In [ ]:
import importlib.util, subprocess, sys

PAQUETES = [("requests", "requests"), ("pandas", "pandas"), ("pyarrow", "pyarrow"), ("matplotlib", "matplotlib"), ("openpyxl", "openpyxl")]   # (módulo, paquete de pip)
faltan = [p for m, p in PAQUETES if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Librerías listas" + (f"; se instalaron: {', '.join(faltan)}" if faltan else ""))

## 2. Parámetros que usted puede ajustar

Los nombres de departamento y tipo de vehículo pueden escribirse con o sin tildes; el cuaderno busca cómo están escritos en la base.

> **Token opcional.** datos.gov.co funciona sin clave, pero con un *app token* gratuito las descargas grandes fallan menos: cree una cuenta en datos.gov.co → *Developer Settings* → *Create New App Token*, y antes de ejecutar escriba `os.environ["SOCRATA_APP_TOKEN"] = "su_token"`.

In [ ]:
DEPARTAMENTO  = "Meta"          # departamento del accidente. None = todo el país (≈ 450 mil vehículos)
ANIOS         = [2024, 2025]    # años entre 2022 y 2025. None = todos
TIPO_VEHICULO = None            # p. ej. "Motocicleta". None = todos
MAX_FILAS     = None            # tope de filas, p. ej. 50_000. None = todas las que cumplan el filtro

FORMATO = "parquet"          # "parquet" (liviano y rápido), "csv" o "excel"
BAJAR_AL_COMPUTADOR = False  # en Colab, True descarga el archivo a su computador al terminar

## 3. Funciones de apoyo

No necesita cambiarlas. Descargan por páginas, reintentan si la conexión falla, resumen la base y la guardan en el formato elegido.

In [ ]:
import os, re, time, unicodedata, requests, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

DATOS = Path("datos"); DATOS.mkdir(exist_ok=True)   # aquí quedan los archivos descargados
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

def sin_tildes(texto):
    """'Bogotá, D.C.' -> 'BOGOTA D C': mayúsculas, sin tildes ni signos."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().upper()
    return " ".join(re.sub(r"[^A-Z0-9 ]", " ", t).split())

def etiqueta(*partes):
    """('saber11', 'Meta') -> 'saber11_meta' (para nombrar archivos)."""
    return "_".join(sin_tildes(p).lower().replace(" ", "_") for p in partes if p)

def entero(x, _=None):
    """12345.6 -> '12.346' (separador de miles colombiano)."""
    return f"{x:,.0f}".replace(",", ".")

def decimal(x, d=1):
    """3.14159 -> '3,1' (coma decimal)."""
    return f"{x:.{d}f}".replace(".", ",")

def ficha_rapida(df):
    """Tamaño, tipo, faltantes, valores distintos y un ejemplo de cada columna."""
    print(f"{entero(len(df))} filas × {df.shape[1]} columnas | "
          f"{decimal(df.memory_usage(deep=True).sum() / 1e6)} MB en memoria")
    tabla = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% faltante": (df.isna().mean() * 100).round(1),
        "valores distintos": df.nunique(),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    display(tabla)

def guardar(df, nombre, formato=None, bajar=None):
    """Guarda la base en datos/ como parquet, csv o excel."""
    formato = (formato or FORMATO).lower()
    if formato == "excel" and len(df) > 1_048_000:
        print("Excel admite como máximo 1.048.576 filas: se guarda en parquet.")
        formato = "parquet"
    ext = {"parquet": "parquet", "csv": "csv", "excel": "xlsx"}[formato]
    ruta = DATOS / f"{nombre}.{ext}"
    sin_zona = df.copy()
    for c in sin_zona.select_dtypes("datetimetz"):
        sin_zona[c] = sin_zona[c].dt.tz_localize(None)
    if formato == "parquet": sin_zona.to_parquet(ruta, index=False)
    elif formato == "csv":   sin_zona.to_csv(ruta, index=False, encoding="utf-8-sig")
    else:                    sin_zona.to_excel(ruta, index=False)
    print(f"Guardado: {ruta} ({entero(len(df))} filas × {df.shape[1]} columnas)")
    if (BAJAR_AL_COMPUTADOR if bajar is None else bajar):
        try:
            from google.colab import files
            files.download(str(ruta))
        except ImportError:
            print("BAJAR_AL_COMPUTADOR solo funciona en Google Colab; el archivo ya está en la carpeta datos/.")
    return ruta

# ---------- datos.gov.co (API Socrata, consultas en lenguaje SoQL) ----------
BASE = "https://www.datos.gov.co/resource/{}.json"
TOKEN = os.getenv("SOCRATA_APP_TOKEN")      # opcional: os.environ["SOCRATA_APP_TOKEN"] = "su_token"
HEADERS = {"X-App-Token": TOKEN} if TOKEN else {}

def socrata(dataset_id, select=None, where=None, group=None, order=None,
            max_rows=None, limit=50_000, timeout=300, reintentos=3, mensajes=True):
    """Descarga un conjunto de datos.gov.co por páginas, con reintentos. Devuelve un DataFrame."""
    if max_rows: limit = min(limit, max_rows)
    params = {"$limit": limit}
    if select: params["$select"] = select
    if where:  params["$where"]  = where
    if group:  params["$group"]  = group
    params["$order"] = order or (group if group else ":id")    # orden estable para paginar
    partes, offset = [], 0
    while True:
        params["$offset"] = offset
        for intento in range(reintentos):
            try:
                r = requests.get(BASE.format(dataset_id), params=params, headers=HEADERS, timeout=timeout)
                r.raise_for_status()
                break
            except requests.RequestException as e:
                if intento == reintentos - 1:
                    raise
                print(f"  reintento {intento + 1}: {e}")
                time.sleep(5 * (intento + 1))
        lote = r.json()
        if not lote:
            break
        partes.append(pd.DataFrame(lote))
        offset += len(lote)
        if mensajes:
            print(f"  {dataset_id}: {entero(offset)} filas descargadas", end="\r")
        if len(lote) < limit or (max_rows and offset >= max_rows):
            break
        time.sleep(0.3)
    if mensajes:
        print()
    df = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
    return df.head(max_rows) if max_rows else df

def contar(dataset_id, where=None):
    """Cuántas filas cumplen el filtro, sin descargarlas."""
    r = socrata(dataset_id, select="count(*) as n", where=where, order="n", mensajes=False)
    return int(r["n"].iloc[0]) if len(r) else 0

def lista_soql(valores):
    """['a', "b'c"] -> ('a', 'b''c') para usar con IN en SoQL."""
    return "(" + ", ".join("'" + str(v).replace("'", "''") + "'" for v in valores) + ")"

def canon(texto):
    """Nombre de departamento comparable entre fuentes (con o sin tildes, 'Bogotá D.C.', 'San Andrés…')."""
    t = sin_tildes(texto)
    if "BOGOTA" in t: return "BOGOTA"
    if "SAN ANDRES" in t: return "SAN ANDRES"
    return t.replace("DEPARTAMENTO DE ", "")

def valores_de(dataset_id, campo, texto, where=None, contiene=False):
    """Busca cómo está escrito `texto` en la columna `campo` (cada fuente lo escribe distinto)."""
    df = socrata(dataset_id, select=campo, where=where, group=campo, mensajes=False)
    if df.empty:
        return []
    todos = df[campo].dropna().unique()
    if contiene:
        hallados = [v for v in todos if sin_tildes(texto) in sin_tildes(v)]
    else:
        hallados = [v for v in todos if canon(v) == canon(texto)]
    if not hallados:
        raise ValueError(f"No encontré '{texto}' en {campo}. Valores posibles: {sorted(todos)}")
    return hallados

def cod5(s):
    """Código DANE de municipio a 5 dígitos como texto (algunas fuentes traen 8 o pierden el cero)."""
    s = pd.Series(s).astype(str).str.replace(r"\.0$", "", regex=True).str.replace(r"\D", "", regex=True)
    return pd.Series(np.where(s.str.len() >= 7, s.str.zfill(8).str[:5], s.str.zfill(5)), index=s.index)

def a_numero(df, cols):
    for c in cols:
        if c in df: df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

In [ ]:
# Estilo gráfico común del curso (paleta validada para daltonismo)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

AZUL, NARANJA, AQUA, ROJO = '#2a78d6', '#eb6834', '#1baf7a', '#e34948'
GRIS, TINTA, TINTA2 = '#898781', '#0b0b0b', '#52514e'

mpl.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'savefig.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': TINTA2, 'text.color': TINTA,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'axes.grid': True, 'grid.color': '#e1e0d9',
    'grid.linewidth': 0.6, 'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'figure.dpi': 110, 'lines.linewidth': 2, 'legend.frameon': False,
})

## 4. ¿Cuántos vehículos cumplen el filtro?

In [ ]:
ACC = "6jmc-vaxk"
filtros = []
if DEPARTAMENTO:
    filtros.append(f"departamento_accidente IN {lista_soql(valores_de(ACC, 'departamento_accidente', DEPARTAMENTO))}")
if ANIOS:
    filtros.append("(" + " OR ".join(f"fecha_accidente like '%/{a}'" for a in ANIOS) + ")")
if TIPO_VEHICULO:
    filtros.append(f"tipo_vehiculo IN {lista_soql(valores_de(ACC, 'tipo_vehiculo', TIPO_VEHICULO))}")
WHERE = " AND ".join(filtros) or None
print("Filtro:", WHERE)
print(f"Vehículos que cumplen el filtro: {entero(contar(ACC, WHERE))}")

## 5. Descarga

In [ ]:
acc = socrata(ACC, where=WHERE, max_rows=MAX_FILAS)
print(f"Descargadas {entero(len(acc))} filas")

## 6. Limpieza mínima

La fecha viene como texto `MM/AAAA`; se convierte y se crea la variable objetivo binaria `con_muertos`.

In [ ]:
acc["fecha"] = pd.to_datetime(acc["fecha_accidente"], format="%m/%Y", errors="coerce")
acc["anio"], acc["mes"] = acc["fecha"].dt.year, acc["fecha"].dt.month
acc = a_numero(acc, ["modelo_vehiculo", "edad_vehiculo"])
acc["con_muertos"] = (acc["gravedad_accidente"] == "CON MUERTOS").astype(int)
acc = acc.drop(columns="fecha_accidente")

## 7. Revisión de la base

In [ ]:
ficha_rapida(acc)
acc.head()

*(Se completa al ejecutar.)*

In [ ]:
tipos = (acc.groupby("tipo_vehiculo")["con_muertos"].agg(vehiculos="size", pct_con_muertos="mean")
            .query("vehiculos >= 30").sort_values("vehiculos").tail(8))
tipos["pct_con_muertos"] *= 100

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
a1.barh(tipos.index, tipos["vehiculos"], color=AZUL); a1.set_title("Vehículos involucrados")
a2.barh(tipos.index, tipos["pct_con_muertos"], color=NARANJA); a2.set_title("% en accidentes con muertos")
plt.tight_layout(); plt.show()
print(f"En toda la base descargada: {decimal(acc['con_muertos'].mean() * 100)} % con muertos")

*(Se completa al ejecutar.)*

## 8. Guardar

In [ ]:
ruta = guardar(acc, etiqueta("accidentes_vehiculos", DEPARTAMENTO or "pais"))

## Siguientes pasos

1. Cargue la base en otro cuaderno: `df = pd.read_parquet("datos/accidentes_vehiculos_meta.parquet")` (o `read_csv` / `read_excel` según el `FORMATO`).
2. Defina la **pregunta** y la **variable objetivo** (vea «Ideas para un modelo de machine learning»).
3. Revise faltantes, valores extremos y tipos de variables antes de modelar (clase 2.1, ingeniería de características).
4. Parta en entrenamiento y prueba **antes** de transformar, y compare su modelo con una **línea base** sencilla.